# This document Combines and compared Controller and Robot Classes

**Merged (2026-11-13):** Dana's waypoint benchmark and Henry's test runs in one notebook, with one runner (`run_simulation`) and one controller list (`CONTROLLERS`). The classes are in `ControllerModels.ipynb`; the checks of the models and the test paths are in `ModelChecks.ipynb`.

* **Frames:** every model's velocity state is global (Henry's convention): Newton's law holds as it is, with no rotational term. The velocity along the robot's axes is worked out only where a wheel needs it.
* **Current:** every MPC keeps |i1| + |i2| + |i3| + |i4| <= `TOTAL_CURRENT_LIMIT` = 7.4 A at every predicted step (the LQR clips at it): the Pololu total at which the robot's BLDCs, doing the same motion, would draw their 15 A from the battery (notes/2026-11-13/CURRENT_LIMIT_DERIVATION.md). The plant itself limits at 15 A (`I_total_max`), for now.\n* **Waypoints, two analyses:** (1) Dana's fixed time (sections 1 to 4, "who is better in the same time"): every waypoint path has a hold at each waypoint, and a waypoint counts only if the robot gets there AND settles: within 3 cm, below 1 cm/s and within 10° of heading, staying so until the end of its hold (X: not settled). (2) Henry's waypoint-executed run (section 5, `WaypointExecution`, "who gets there first"): the controller drives the reference's progress, which waits at each waypoint until it settles (no hold) and on the eights at each marker until it is within 3 cm; the run bottoms out at the scheduled end plus `WAYPOINT_TIME_ALLOWANCE`. No deadband: the controller has to hold it.
* **Cost weights:** one set for every controller with a cost (the MPCs and the LQR): `COST_WEIGHTS` in the settings, Henry's Bryson rule or Dana's set. Every MPC's terminal cost is one setting too (`TERMINAL_COST`: Riccati or fixed).


In [ ]:
import time

class CodeTimer:
    """
    A versatile timer class for measuring elapsed execution time with 
    support for starting, pausing, resuming, and resetting.
    """
    def __init__(self):
        self.reset()

    def start(self):
        """Starts the timer from a fresh state or restarts it."""
        self._start_time = time.perf_counter()
        self._elapsed_time = 0.0
        self._is_running = True

    def pause(self):
        """Pauses the timer if it is currently running."""
        if self._is_running:
            self._elapsed_time += time.perf_counter() - self._start_time
            self._is_running = False

    def resume(self):
        """Resumes the timer if it was paused."""
        if not self._is_running:
            self._start_time = time.perf_counter()
            self._is_running = True

    def reset(self):
        """Resets all timer counters and stops the timer."""
        self._start_time = 0.0
        self._elapsed_time = 0.0
        self._is_running = False

    @property
    def elapsed(self) -> float:
        """Returns the total elapsed time in seconds, updating live if running."""
        if self._is_running:
            return self._elapsed_time + (time.perf_counter() - self._start_time)
        return self._elapsed_time


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# =======================================================================
# 0. BRING IN FUNCTIONS FROM OTHER NOTEBOOKS
# =======================================================================

from IPython.utils.capture import capture_output
global SIMULATION_PARAMATERS, PathGenerator, OmniRobotSimulationExplicit
# This forces the native Jupyter runner to execute the notebook silently
with capture_output():
    %run "PlantModels.ipynb"
    %run "ControllerModels.ipynb"
    %run "PathGeneration.ipynb"


## Settings, controllers and the runner


In [ ]:
# paramaters and classed are defined in xxxx.ipynb and imported above, so we need to declare them as global to avoid "undefined variable" errors
global model_choice, SIMULATION_PARAMATERS, PathGenerator, PathGeneratorWithHold
global OmniRobotSimulationExplicit, OmniRobotStateExplicit, OmniRobotSimulationODESolve, OmniRobotStateODESolve
global CascadedPIDController, MPCControllerSimple
global LQRController, LumpedDOBMPCController, SaturatedCoulombMPCController, LuGreBristleMPCController
global MPCControllerLinearized, PathGeneratorCurvedWithHold, CodeTimer   # MERGED: the classes used below


In [ ]:
# =======================================================================
# SETTINGS AND CONTROLLERS (shared by every test below)
# =======================================================================
p = SIMULATION_PARAMATERS
DT_CTRL = 0.02
DT_PLANT = p['dt']
HOLD_DURATION = 0.5             # s: the hold at each waypoint (Dana's; Henry's waypoint-executed run uses none)

# Settled at a waypoint (Dana's thresholds): inside all three, and staying inside until the path moves on
ARRIVE_THRESHOLD_POS = 0.03     # 3.0 cm
ARRIVE_THRESHOLD_VEL = 0.01     # 1.0 cm/s
ARRIVE_THRESHOLD_THETA = 10     # deg

# One set of cost weights for every controller with a cost (the MPCs and the LQR)
COST_WEIGHTS = 'bryson'         # 'bryson' (Henry) or 'dana'
if COST_WEIGHTS == 'bryson':
    # Bryson's rule: weight = 1 / (largest acceptable value)^2
    max_pos_err = 0.05                      # m
    max_heading_err = np.radians(5.0)       # rad
    max_vel_err = 0.10                      # m/s
    max_dV = 0.5                            # V per 20 ms step: the main voltage cost
    # The light cost on the voltage itself: the voltage that holds the top wheel speed and pushes at the grip limit
    w_top = 40.0                            # rad/s
    i_grip = p['mu_s'] * p['Mr'] * p['g'] / 4 * p['r'] / p['Kt']
    max_V = p['Kb'] * w_top + p['R'] * i_grip
    HORIZON = 8
    Q_POSITION = (1 / max_pos_err**2, 1 / max_pos_err**2, 1 / max_heading_err**2)
    Q_VELOCITY = (1 / max_vel_err**2, 1 / max_vel_err**2, 1 / (max_vel_err / p['Rb'])**2)
    R_VOLTAGE = 1 / max_V**2
    R_SLEW = 1 / max_dV**2
    DELTA_V_LIMIT = 5.0
    P_POSITION = None                       # terminal weights: P_MULTIPLY x the stage weights
    P_VELOCITY = None
else:
    # Dana's set (her ControllerAnalysis, 11-13)
    HORIZON = 5
    Q_POSITION = (101.0, 101.0, 60.0)
    Q_VELOCITY = (1.0, 1.0, 1.0)
    R_VOLTAGE = 0.001
    R_SLEW = 0.001
    DELTA_V_LIMIT = 1
    P_POSITION = (1000.0, 1000.0, 1000.0)
    P_VELOCITY = None
P_MULTIPLY = 10
# Every MPC's terminal cost (RiccatiTerminalCost): 'riccati' (the weight on the last step from the discrete Riccati
# equation of that step's model, at every solve) or 'fixed' (P_MULTIPLY x the stage weights, or P_POSITION / P_VELOCITY)
TERMINAL_COST = 'riccati'
# Every MPC: |i1| + |i2| + |i3| + |i4| <= TOTAL_CURRENT_LIMIT at every predicted step (A); the LQR clips at it. 7.4 A: the
# Pololu total at which the robot's BLDCs, doing the same motion, would draw their 15 A from the 16.8 V battery
# (notes/2026-11-13/CURRENT_LIMIT_DERIVATION.md). The plant limits at its own I_total_max (15 A for now). None: off
TOTAL_CURRENT_LIMIT = 7.4

mpc_settings = dict(horizon_K=HORIZON, dt=DT_CTRL, q_pose=Q_POSITION, q_vel=Q_VELOCITY, r_voltage=R_VOLTAGE, r_slew=R_SLEW,
                    voltage_limit=p['V_max'], current_limit=p['I_max'], delta_v_max=DELTA_V_LIMIT,
                    p_multiply=P_MULTIPLY, p_pose_fin=P_POSITION, p_vel_fin=P_VELOCITY, terminal_cost=TERMINAL_COST,
                    total_current_limit=TOTAL_CURRENT_LIMIT)
disturbance_settings = dict(horizon_K=HORIZON, dt=DT_CTRL, q_pose=Q_POSITION, q_vel=Q_VELOCITY, r_voltage=R_VOLTAGE, r_slew=R_SLEW,
                            voltage_limit=p['V_max'], delta_v_max=DELTA_V_LIMIT, terminal_cost=TERMINAL_COST,
                            total_current_limit=TOTAL_CURRENT_LIMIT)

# Every controller: name -> (a function that builds a fresh one, plot colour)
CONTROLLERS = {
    'PID':                     (lambda: CascadedPIDController(p, pos_pid_gains={'kp': 3.0, 'ki': 0.2, 'kd': 0.05},
                                                              wheel_pid_gains={'kp': 0.15, 'ki': 0.8, 'kd': 0.0}, dt=DT_CTRL,
                                                              a_max=15, alpha_max=10.0, k_theta=3.0, voltage_limit=p['V_max']), 'tab:red'),
    'PID tuned':               (lambda: CascadedPIDController(p, pos_pid_gains={'kp': 3, 'ki': 0.1, 'kd': 1},
                                                              wheel_pid_gains={'kp': 0.05, 'ki': 20, 'kd': 0.0}, dt=DT_CTRL,
                                                              a_max=1e6, alpha_max=1e6, k_theta=25, voltage_limit=p['V_max']), 'tab:purple'),
    'Simple MPC':              (lambda: MPCControllerSimple(p, **mpc_settings), 'tab:blue'),
    'Simple MPC no vel':       (lambda: MPCControllerSimple(p, **dict(mpc_settings, q_vel=(0.0, 0.0, 0.0))), 'tab:cyan'),
    'LQR':                     (lambda: LQRController(p, dt=DT_CTRL, q_pose=Q_POSITION, q_vel=Q_VELOCITY, r_voltage=R_VOLTAGE,
                                                      voltage_limit=p['V_max'], delta_v_max=DELTA_V_LIMIT,
                                                      total_current_limit=TOTAL_CURRENT_LIMIT), 'tab:green'),
    'DOB MPC':                 (lambda: LumpedDOBMPCController(p, **disturbance_settings, p_pose=P_POSITION, p_vel=P_VELOCITY), 'tab:brown'),
    'Coulomb MPC':             (lambda: SaturatedCoulombMPCController(p, **disturbance_settings, p_pose=P_POSITION, p_vel=P_VELOCITY), 'tab:gray'),
    'LuGre MPC':               (lambda: LuGreBristleMPCController(p, **disturbance_settings), 'tab:olive'),
    'Linearized MPC':          (lambda: MPCControllerLinearized(p, state_estimate='pose', **mpc_settings), 'tab:orange'),
    'Linearized MPC observer': (lambda: MPCControllerLinearized(p, state_estimate='observer', **mpc_settings), 'tab:pink'),
    'Linearized MPC true':     (lambda: MPCControllerLinearized(p, state_estimate='true', **mpc_settings), 'black'),
}
print(f"Cost weights: {COST_WEIGHTS}, horizon {HORIZON}; terminal cost: {TERMINAL_COST}; total current limit: {TOTAL_CURRENT_LIMIT} A; {len(CONTROLLERS)} controllers")


In [ ]:
# =======================================================================
# THE RUNNER (one for every test) AND THE HELPERS
# =======================================================================
from matplotlib.patches import Circle, Patch


WAYPOINT_TIME_ALLOWANCE = 1.0     # s: Henry's waypoint-executed run ends this long after the path's scheduled end


def is_settled(pos, v_local, waypoint):
    """Dana's settle test: within ARRIVE_THRESHOLD_POS, below ARRIVE_THRESHOLD_VEL and within ARRIVE_THRESHOLD_THETA degrees of the waypoint."""
    distance = np.hypot(*(pos[:2] - waypoint[:2]))
    speed = np.hypot(*v_local[:2])
    heading_err = np.degrees(np.abs((pos[2] - waypoint[2] + np.pi) % (2.0 * np.pi) - np.pi))
    return distance <= ARRIVE_THRESHOLD_POS and speed <= ARRIVE_THRESHOLD_VEL and heading_err <= ARRIVE_THRESHOLD_THETA


class WaypointExecution:
    """
    Henry's waypoint-executed run ("who gets there first"), to pass to run_simulation(..., executor=...). Without it,
    run_simulation plays the path on its fixed timeline (Dana's: "who is better in the same time").

    The controller drives the reference's progress: the reference plays the path, but
      at a waypoint (stop): it waits at the end of the waypoint's hold until the robot has settled there (is_settled);
      at a marker (the figure eights, pass-through): it waits at the marker until the robot is within
        ARRIVE_THRESHOLD_POS of it (no need to stop), then carries on along the path.
    Until then the controller sees the reference only up to that point, not beyond it. The run bottoms out at a fixed
    time, the path's scheduled end plus time_allowance: a slow controller tracks the path but does not reach the points
    it has not reached by then.
    """
    def __init__(self, time_allowance: float = WAYPOINT_TIME_ALLOWANCE):
        self.time_allowance = time_allowance

    def start(self, path: dict) -> int:
        """Sets up the points the reference waits at, in path order; returns the step at which the run ends."""
        self.stops = []
        if 'waypoints' in path:
            for wp in path['waypoints'][1:]:
                self.stops.append((get_hold_steps(path, wp)[1], 'settle', wp))
        if 'markers' in path:
            for marker, step in zip(path['markers'], path['marker_steps']):
                self.stops.append((step, 'pass', marker))
        self.stops.sort(key=lambda stop: stop[0])
        self.next_stop = 0
        return len(path['pos']) + int(round(self.time_allowance / DT_CTRL))

    def last_seen(self, r: int, N: int) -> int:
        """The last path step the controller may see: the point the reference waits at next (or the end of the path)."""
        if self.next_stop < len(self.stops):
            return self.stops[self.next_stop][0]
        return N - 1

    def next_step(self, r: int, state) -> int:
        """The reference's next path step: it stays at a stop until the robot has settled there (or passed within reach)."""
        if self.next_stop < len(self.stops) and r == self.stops[self.next_stop][0]:
            step, kind, point = self.stops[self.next_stop]
            if kind == 'settle':
                reached = is_settled(state.pos_global, state.v_local, point)
            else:
                reached = np.hypot(*(state.pos_global[:2] - point[:2])) <= ARRIVE_THRESHOLD_POS
            if not reached:
                return r
            self.next_stop += 1
        return r + 1


def run_simulation(controller_name: str, path: dict, executor: WaypointExecution = None) -> dict:
    """
    Runs one controller on one path with the ODESolve plant: the controller every 20 ms, the plant every 1 ms.
    The MPCs get the GLOBAL velocity reference (their velocity state is global). The 1 ms observer gets the readings
    every plant step; 'Linearized MPC true' gets the plant's true state (a test only).

    Dana's fixed timeline by default: the reference plays the path in time, the same for every controller. With an
    executor (WaypointExecution, Henry's), the controller drives the reference's progress instead, so each controller
    has its own timeline.

    Returns Dana's result keys and Henry's, plus the settled time at each waypoint (None: not settled), each
    waypoint's (arrive, release) steps on this controller's timeline (None: the reference never got there), and on a
    path with markers the step at which the reference reached each one (marker_steps) and when the robot got within
    reach of it (marker_reached, None: never).
    """
    controller = CONTROLLERS[controller_name][0]()
    plant = OmniRobotSimulationODESolve(SIMULATION_PARAMATERS, random_seed=42)
    state = OmniRobotStateODESolve()
    substeps = int(round(DT_CTRL / DT_PLANT))

    path_pos = path['pos']
    path_vel = path['vel']
    N = len(path_pos)

    waypoints = path['waypoints'][1:] if 'waypoints' in path else []
    if executor is None:
        end_step = N                         # Dana's fixed timeline
    else:
        end_step = executor.start(path)      # Henry's: the run bottoms out here

    is_pid = isinstance(controller, CascadedPIDController)
    horizon = 1 if (is_pid or isinstance(controller, LQRController)) else controller.K
    feeds_observer = getattr(controller, 'state_estimate', None) == 'observer'

    log_pos, log_v_local, log_v_global, log_V, log_I, log_slip, log_z, log_opt = [], [], [], [], [], [], [], []
    log_V_1ms, log_I_1ms, log_step = [], [], []
    timer = CodeTimer()
    timer.start()
    r = 0               # the path step the reference is at
    while r < N and len(log_step) < end_step:
        obs = plant.get_observable_state(state)

        # The reference over the horizon (Dana's), held from the end of the path (or, with an executor, the point it waits at next)
        if executor is None:
            last_seen = N - 1
        else:
            last_seen = executor.last_seen(r, N)
        steps_ahead = np.arange(r, r + horizon + 1)
        pos_h = path_pos[np.minimum(steps_ahead, last_seen)].copy()
        vel_h = path_vel[np.minimum(steps_ahead, last_seen)].copy()
        vel_h[steps_ahead > last_seen] = 0.0

        if is_pid:
            u = controller.compute(path_pos[r], obs)
            optimal = True
        else:
            if controller_name == 'Linearized MPC true':
                controller.true_state = state
            u, info = controller.compute_optimal_u0(obs, pos_h, vel_h, return_info=True)
            optimal = info['optimal']

        # The plant every 1 ms (not timed)
        timer.pause()
        for q in range(substeps):
            state = plant.step(state, u)
            log_V_1ms.append(state.V_actual.copy())
            log_I_1ms.append(state.I.copy())
            if feeds_observer:
                controller.estimator_update(plant.get_observable_state(state), camera_frame=(q == substeps - 1))

        log_pos.append(state.pos_global.copy())
        log_v_local.append(state.v_local.copy())
        log_v_global.append(state.v_global.copy())
        log_V.append(u.copy())
        log_I.append(state.I.copy())
        log_slip.append(plant.H @ state.v_local - p['r'] * state.omega)
        log_z.append(state.z.copy())
        log_opt.append(optimal)
        log_step.append(r)

        # The next reference step: the next in time (Dana's), or the executor's
        if executor is None:
            r += 1
        else:
            r = executor.next_step(r, state)
        timer.resume()
    timer.pause()

    # This controller's timeline: the path step the reference was at, each control step
    log_step = np.array(log_step)
    t = DT_CTRL * np.arange(len(log_step))
    pos_ref = path_pos[log_step]
    vel_ref = path_vel[log_step]
    waypoint_steps = []
    for wp in waypoints:
        first, last = get_hold_steps(path, wp)
        if np.any(log_step == first):
            arrive = np.flatnonzero(log_step == first)[0]
            if np.any(log_step == last):
                release = np.flatnonzero(log_step == last)[-1]
            else:
                release = len(log_step) - 1          # the run ended during its hold
            waypoint_steps.append((arrive, release))
        else:
            waypoint_steps.append(None)              # the reference never got there

    # Markers (the figure eights): when the reference reached each one, and when the robot got within reach of it
    marker_steps, marker_reached = [], []
    if 'markers' in path:
        for marker, step in zip(path['markers'], path['marker_steps']):
            if np.any(log_step == step):
                marker_steps.append(np.flatnonzero(log_step == step)[0])
                within = np.hypot(*(np.array(log_pos)[marker_steps[-1]:, :2] - marker[:2]).T) <= ARRIVE_THRESHOLD_POS
                if np.any(within):
                    marker_reached.append(t[marker_steps[-1] + np.flatnonzero(within)[0]] + DT_CTRL)
                else:
                    marker_reached.append(None)
            else:
                marker_steps.append(None)
                marker_reached.append(None)

    log_pos = np.array(log_pos)
    log_v_local = np.array(log_v_local)
    dist_err = np.hypot(pos_ref[:, 0] - log_pos[:, 0], pos_ref[:, 1] - log_pos[:, 1])
    th_err = np.abs((pos_ref[:, 2] - log_pos[:, 2] + np.pi) % (2.0 * np.pi) - np.pi)
    settled = get_settled_times(waypoints, waypoint_steps, t, log_pos, log_v_local)
    log_I_1ms = np.array(log_I_1ms)
    return {
        # Dana's keys
        't': t, 'pos': log_pos, 'v_local': log_v_local, 'V_cmd': np.array(log_V), 'I': np.array(log_I),
        'v_slip': np.array(log_slip), 'z': np.array(log_z), 'pos_err': dist_err,
        'arrival_times': [0.0] + settled, 'exec_time': timer.elapsed,
        # Henry's keys
        'pos_ref': pos_ref, 'vel_ref': vel_ref, 'v': np.array(log_v_global),
        'V': np.array(log_V), 'vs': np.array(log_slip), 'dist_err': dist_err, 'th_err': th_err,
        'log_opt': np.array(log_opt), 'elapsed': timer.elapsed,
        't_1ms': DT_PLANT * np.arange(1, len(log_I_1ms) + 1), 'V_1ms': np.array(log_V_1ms), 'I_1ms': log_I_1ms,
        'settled': settled, 'waypoint_steps': waypoint_steps, 'marker_steps': marker_steps, 'marker_reached': marker_reached,
    }


def get_settled_times(waypoints, waypoint_steps, t, pos, v_local) -> list:
    """
    When the robot settled at each waypoint (None: not attained), on the controller's own timeline: settled
    (is_settled) and staying so until the path moved on from the waypoint (or the run ended). Waypoint i's window runs
    from the step after the path left waypoint i-1 to the step it left waypoint i (waypoint_steps: (arrive, release)
    steps, None if the reference never got there); the settled time is when the last stretch inside the thresholds
    began. A path without waypoints (the figure eights) gives [].
    """
    t_end_of_step = t + DT_CTRL                  # the plant state is logged at the end of each 20 ms step
    settled = []
    window_start = 0
    for wp, steps in zip(waypoints, waypoint_steps):
        if steps is None:
            settled.append(None)
            continue
        arrive, release = steps
        window = np.arange(window_start, release + 1)
        inside = np.array([is_settled(pos[k], v_local[k], wp) for k in window])
        if inside[-1]:
            # The start of the last stretch inside: walk back from the release while the robot is still inside
            first = len(inside) - 1
            while first > 0 and inside[first - 1]:
                first -= 1
            settled.append(t_end_of_step[window[first]])
        else:
            settled.append(None)
        window_start = release + 1
    return settled


def path_errors(robot_xy, reference_xy, search_back=10, search_ahead=50):
    """
    The tracking error split into its two parts (reporting only; the controllers are unchanged):
      perpendicular error: the shortest distance from the robot to the track, at right angles to it; time ignored;
      lag error: how far along the track the robot is behind the reference at the same step (negative: ahead).
    The points where the reference stands still (the holds) are dropped first. Each step, the robot is projected onto
    the nearest piece of the track, searched near the previous projection (search_back pieces back, search_ahead
    ahead) and always including the reference's own piece at that step: so it does not jump to another loop where a
    path crosses itself, and the perpendicular error is never more than the position error at that step.
    Returns perpendicular (N,), lag (N,) in metres.
    """
    step = np.hypot(*np.diff(reference_xy, axis=0).T)
    s_of_step = np.concatenate(([0.0], np.cumsum(step)))       # arc length of the reference at each step
    moving = np.concatenate(([True], step > 1e-9))             # drop the points where it stands still
    points = reference_xy[moving]
    s_points = s_of_step[moving]
    point_of_step = np.cumsum(moving) - 1                      # the kept point each step's reference sits on
    a = points[:-1]
    ab = points[1:] - a
    seg_len = np.maximum(np.hypot(ab[:, 0], ab[:, 1]), 1e-12)
    perpendicular = np.zeros(len(robot_xy))
    lag = np.zeros(len(robot_xy))
    j_prev = 0
    for k, robot in enumerate(robot_xy):
        j_ref = min(point_of_step[k], len(a) - 1)
        j = np.arange(max(0, min(j_prev, j_ref) - search_back), min(len(a), max(j_prev, j_ref) + search_ahead + 1))
        u = np.clip(np.sum((robot - a[j]) * ab[j], axis=1) / seg_len[j]**2, 0.0, 1.0)
        nearest = a[j] + u[:, None] * ab[j]
        d = np.hypot(*(nearest - robot).T)
        i = int(np.argmin(d))
        j_prev = int(j[i])
        perpendicular[k] = d[i]
        lag[k] = s_of_step[k] - (s_points[j_prev] + u[i] * seg_len[j_prev])
    return perpendicular, lag


def run_and_report(title, paths, controllers, executor=None):
    """
    Run every controller on every path; print one table (Henry's columns, and Dana's settled time at each waypoint);
    return the results.
    """
    results = {}
    print(f"{title}\n{'Path':34s} {'Controller':24s} {'Off path (cm)':>13s} {'Perp (cm)':>9s} {'Lag (cm)':>8s} "
          f"{'Max (cm)':>8s} {'RMS (cm)':>8s} {'Final (cm)':>10s} {'Heading (deg)':>13s} {'Sliding (%)':>11s} {'Peak |V|':>8s} "
          f"{'Energy (J)':>10s} {'ms/step':>7s}   Settled at (s), each waypoint (X: not settled)")
    for path_name, path in paths.items():
        for controller in controllers:
            res = run_simulation(controller, path, executor=executor)
            results[(path_name, controller)] = res
            off_path = largest_distance_off_path(res['pos'][:, :2], res['pos_ref'][:, :2])
            res['perp_err'], res['lag_err'] = path_errors(res['pos'][:, :2], res['pos_ref'][:, :2])
            sliding = 100 * np.mean(np.any(np.abs(res['vs']) > p['slip_band'], axis=1))
            if res['settled']:
                settled_text = " ".join("X" if t is None else f"{t:.2f}" for t in res['settled'])
            elif 'markers' in path and executor is not None:
                n_reached = sum(r is not None for r in res['marker_reached'])
                settled_text = f"(markers reached: {n_reached}/{len(path['markers'])}, run ended at {res['t'][-1] + DT_CTRL:.2f} s)"
            elif 'markers' in path:
                marker_miss = np.hypot(*(res['pos'][path['marker_steps'], :2] - path['markers'][:, :2]).T)
                settled_text = f"(markers: mean {100 * marker_miss.mean():.1f} cm, worst {100 * marker_miss.max():.1f} cm off)"
            else:
                settled_text = "(no waypoints)"
            print(f"{path_name:34s} {controller:24s} {100 * off_path:13.1f} {100 * np.sqrt(np.mean(res['perp_err']**2)):9.1f} "
                  f"{100 * np.sqrt(np.mean(res['lag_err']**2)):8.1f} {100 * res['dist_err'].max():8.1f} {100 * np.sqrt(np.mean(res['dist_err']**2)):8.1f} "
                  f"{100 * res['dist_err'][-1]:10.1f} {np.degrees(res['th_err'].max()):13.1f} {sliding:11.1f} {np.abs(res['V_1ms']).max():8.1f} "
                  f"{np.sum(np.abs(res['V_1ms'] * res['I_1ms'])) * DT_PLANT:10.1f} {1000 * res['elapsed'] / len(res['t']):7.1f}   {settled_text}", flush=True)
    return results


def draw_waypoints(ax, path, arrow_length):
    """
    Dana's waypoint marks on a path plot: each waypoint (red), its settle circle (ARRIVE_THRESHOLD_POS) and its heading.
    A path with markers instead (the figure eights): each marker (red), numbered on the first lap.
    """
    if 'markers' in path:
        for n, marker in enumerate(path['markers']):
            ax.plot(marker[0], marker[1], 'ro', ms=5, zorder=8)
            if n < path['markers_per_lap']:
                ax.text(marker[0], marker[1], f'  M{n + 1}', fontsize=8, fontweight='bold')
        return
    if 'waypoints' not in path:
        return
    for i, wp in enumerate(path['waypoints']):
        ax.plot(wp[0], wp[1], 'ro', ms=6, zorder=8)
        ax.add_patch(Circle((wp[0], wp[1]), ARRIVE_THRESHOLD_POS, color='g', alpha=0.25, zorder=3))
        heading_tip = (wp[0] - np.sin(wp[2]) * arrow_length, wp[1] + np.cos(wp[2]) * arrow_length)
        ax.annotate('', xy=heading_tip, xytext=(wp[0], wp[1]), arrowprops=dict(arrowstyle='->', color='r', lw=1.2))
        ax.text(wp[0], wp[1], f'  WP{i}', fontsize=8, fontweight='bold')


def get_hold_steps(path, waypoint):
    """
    The steps of a waypoint's hold on the path: from when the reference reaches it to the end of its hold (the last
    stretch the reference sits on it, so a waypoint the path also passes earlier, such as the start of a loop, is
    handled). Returns (first, last) step indices.
    """
    on_waypoint = np.hypot(*(path['pos'][:, :2] - waypoint[:2]).T) < 1e-6
    last = np.flatnonzero(on_waypoint)[-1]
    first = last
    while first > 0 and on_waypoint[first - 1]:
        first -= 1
    return first, last


def add_eight_markers(path, size, markers_per_lap=7):
    """
    Markers on a continuous figure eight, a visual check only (the path does not stop or wait at them): markers_per_lap
    points equally spaced along the eight's length, on every lap. Each marker is the reference's sample nearest that
    length, so the plots show where each controller was as the reference passed it.
    """
    # The length of one lap of the shape x = a sin(phi), y = a/2 sin(2 phi)
    phi = np.linspace(0.0, 2.0 * np.pi, 4001)
    lap_length = np.sum(np.hypot(np.diff(size * np.sin(phi)), np.diff(0.5 * size * np.sin(2.0 * phi))))

    # The reference's length along the way, and the steps at which it passes each marker
    travelled = np.concatenate(([0.0], np.cumsum(np.hypot(*np.diff(path['pos'][:, :2], axis=0).T))))
    spacing = lap_length / markers_per_lap
    marker_lengths = spacing * np.arange(1, int(travelled[-1] / spacing) + 1)
    marker_steps = np.searchsorted(travelled, marker_lengths)
    path['marker_steps'] = marker_steps
    path['markers'] = path['pos'][marker_steps]
    path['markers_per_lap'] = markers_per_lap
    return path


def mark_settled_times(axes, results, path_name, controllers):
    """On time plots: when each controller settled at each waypoint, a dotted line in its colour (its own timeline)."""
    for controller in controllers:
        for settled_time in results[(path_name, controller)]['settled']:
            if settled_time is not None:
                for ax in axes:
                    ax.axvline(settled_time, color=CONTROLLERS[controller][1], ls=':', lw=1.0, alpha=0.8)


def draw_arrival_positions(ax, results, path_name, path, controllers):
    """
    Where each controller was when the reference reached each waypoint (or passed each marker): a cross in the
    controller's colour, joined to the waypoint by a dashed line. A controller that falls behind is short of it; one
    that cuts corners shows in its path between them.
    """
    for controller in controllers:
        result = results[(path_name, controller)]
        if 'markers' in path:
            points, steps = path['markers'], result['marker_steps']
        elif 'waypoints' in path:
            points = path['waypoints'][1:]
            steps = [None if wp_steps is None else wp_steps[0] for wp_steps in result['waypoint_steps']]
        else:
            return
        color = CONTROLLERS[controller][1]
        for point, step in zip(points, steps):
            if step is None:
                continue                     # the reference never got there before the run ended
            robot = result['pos'][step]
            ax.plot([point[0], robot[0]], [point[1], robot[1]], color=color, lw=0.8, ls='--', zorder=6)
            ax.plot(robot[0], robot[1], marker='x', color=color, ms=7, mew=2, zorder=9)


def draw_not_attained(ax, result, waypoints, color):
    """
    On a path plot: for each waypoint the controller did not attain, an X in its colour where it was when the path
    moved on from it (the end of its hold, or of the run), joined to that waypoint by a dotted line. Returns True if it
    drew any.
    """
    drew = False
    for wp, settled_time, steps in zip(waypoints[1:], result['settled'], result['waypoint_steps']):
        if settled_time is None:
            if steps is None:
                end = result['pos'][-1]              # the reference never got there: where it was at the end
            else:
                end = result['pos'][steps[1]]
            ax.plot([wp[0], end[0]], [wp[1], end[1]], color=color, lw=1.0, ls=':', zorder=6)
            ax.plot(end[0], end[1], 'X', color=color, mec='k', ms=12, zorder=10)
            drew = True
    return drew


def get_release_miss(result, i, waypoint):
    """
    What was still off at waypoint i (0: the first after the start) when the path moved on, or when the run ended if
    it did not attain it, as short text such as '4.1 cm' or '4.1 cm, 12°' (the settle thresholds it was outside);
    with the distance (m), heading error (deg) and speed (m/s) there.
    """
    if result['waypoint_steps'][i] is None:
        release = len(result['t']) - 1          # the reference never got there: where the robot was at the end
    else:
        release = result['waypoint_steps'][i][1]
    robot = result['pos'][release]
    distance = np.hypot(*(robot[:2] - waypoint[:2]))
    heading_err = np.degrees(np.abs((robot[2] - waypoint[2] + np.pi) % (2.0 * np.pi) - np.pi))
    speed = np.hypot(*result['v_local'][release, :2])
    parts = []
    if distance > ARRIVE_THRESHOLD_POS:
        parts.append(f'{100 * distance:.1f} cm')
    if heading_err > ARRIVE_THRESHOLD_THETA:
        parts.append(f'{heading_err:.0f}°')
    if speed > ARRIVE_THRESHOLD_VEL:
        parts.append(f'{100 * speed:.1f} cm/s')
    return ', '.join(parts), distance, heading_err, speed


def plot_waypoint_closeups(results, paths, controllers, columns=3):
    """
    For each path, a close-up of each waypoint some controller did not attain by the end of the run: where each
    controller was from when the reference arrived until the path moved on or the run ended (thin line, in cm from the
    waypoint), and exactly where it was then (dot; X: not attained), with the settle circle (ARRIVE_THRESHOLD_POS) and
    the waypoint's heading. The legend gives each one's distance, heading error and speed there, and its settled time
    (counted from when the reference arrived).
    """
    for path_name, path in paths.items():
        if 'waypoints' not in path:
            continue
        missed = [i for i in range(1, len(path['waypoints']))
                  if any(results[(path_name, c)]['settled'][i - 1] is None for c in controllers)]
        if not missed:
            continue
        rows = int(np.ceil(len(missed) / columns))
        fig, axs = plt.subplots(rows, columns, figsize=(5.5 * columns, 5 * rows), squeeze=False)
        for ax, i in zip(axs.ravel(), missed):
            wp = path['waypoints'][i]
            reach = 100 * ARRIVE_THRESHOLD_POS
            for controller in controllers:
                result = results[(path_name, controller)]
                if result['waypoint_steps'][i - 1] is None:
                    arrive = release = len(result['t']) - 1     # never got there: where it was at the end
                else:
                    arrive, release = result['waypoint_steps'][i - 1]
                offset = 100 * (result['pos'][arrive:release + 1, :2] - wp[:2])
                text, distance, heading_err, speed = get_release_miss(result, i - 1, wp)
                settled_time = result['settled'][i - 1]
                if settled_time is None:
                    settled_text = 'X: not attained'
                    end_marker = 'X'
                else:
                    settled_text = f'settled +{settled_time - (result["t"][arrive] + DT_CTRL):.2f} s'
                    end_marker = 'o'
                color = CONTROLLERS[controller][1]
                ax.plot(offset[:, 0], offset[:, 1], color=color, lw=1.0, alpha=0.7)
                ax.plot(offset[-1, 0], offset[-1, 1], end_marker, color=color, ms=8, zorder=9,
                        label=f'{controller}: {100 * distance:.1f} cm, {heading_err:.0f}°, {100 * speed:.1f} cm/s, {settled_text}')
                reach = max(reach, np.abs(offset[-1]).max())
            ax.add_patch(Circle((0.0, 0.0), 100 * ARRIVE_THRESHOLD_POS, color='g', alpha=0.2, zorder=1))
            ax.plot(0.0, 0.0, 'r+', ms=12, mew=2, zorder=10)
            ax.annotate('', xy=(-np.sin(wp[2]) * 0.5 * reach, np.cos(wp[2]) * 0.5 * reach), xytext=(0.0, 0.0),
                        arrowprops=dict(arrowstyle='->', color='r', lw=1.2))
            limit = 1.3 * reach
            ax.set_xlim(-limit, limit)
            ax.set_ylim(-limit, limit)
            ax.set_aspect('equal')
            ax.set_xlabel('x from the waypoint (cm)')
            ax.set_ylabel('y from the waypoint (cm)')
            ax.set_title(f'{path_name}: WP{i} (dot: settled, where the path moved on; X: not attained, at the end)', fontsize=10)
            ax.grid(True, linestyle=':', alpha=0.6)
            ax.legend(fontsize='x-small', loc='upper center', bbox_to_anchor=(0.5, -0.12))
        for ax in axs.ravel()[len(missed):]:
            ax.axis('off')
        plt.tight_layout()
        plt.show()


def plot_waypoint_timing(results, paths, controllers, columns=3):
    """
    For each path with waypoints, two panels (on Dana's fixed timeline, or each controller's own with WaypointExecution):
      top: how long it took to settle at each waypoint after the reference arrived. Hatched: not attained; the bar
           runs to the end of its hold or the run (the time it had there). Negative: it settled just before the reference came to rest.
      bottom: where it got to: its distance from the waypoint when the path moved on (or the run ended), against the
           settle radius (dashed). Hatched: not attained, labelled with what was still off then.
    """
    waypoint_paths = [name for name in paths if 'waypoints' in paths[name]]
    rows = int(np.ceil(len(waypoint_paths) / columns))
    fig, axs = plt.subplots(2 * rows, columns, figsize=(6 * columns, 7 * rows), squeeze=False)
    width = 0.8 / len(controllers)
    for n, path_name in enumerate(waypoint_paths):
        ax_time = axs[2 * (n // columns), n % columns]
        ax_dist = axs[2 * (n // columns) + 1, n % columns]
        path = paths[path_name]
        number = np.arange(1, len(path['waypoints']))
        largest_distance = 100 * ARRIVE_THRESHOLD_POS
        for c, controller in enumerate(controllers):
            result = results[(path_name, controller)]
            color = CONTROLLERS[controller][1]
            x = number - 0.4 + width * (c + 0.5)
            times, distances, miss_texts = [], [], []
            for i, settled_time in enumerate(result['settled']):
                text, distance, heading_err, speed = get_release_miss(result, i, path['waypoints'][i + 1])
                steps = result['waypoint_steps'][i]
                if settled_time is not None:
                    times.append(settled_time - (result['t'][steps[0]] + DT_CTRL))   # the state is logged at the end of each step
                    miss_texts.append(None)
                elif steps is None:
                    times.append(0.0)                                             # the reference never got there
                    miss_texts.append(text + ', not reached')
                    ax_time.annotate('not reached', (x[i], 0.0), xytext=(0, 3), textcoords='offset points',
                                     rotation=90, ha='center', va='bottom', fontsize=7)
                else:
                    times.append((steps[1] - steps[0] + 1) * DT_CTRL)             # the time it had there
                    miss_texts.append(text)
                distances.append(100 * distance)
            largest_distance = max(largest_distance, max(distances))
            bars_time = ax_time.bar(x, times, width, color=color, label=controller)
            bars_dist = ax_dist.bar(x, distances, width, color=color, label=controller)
            for bar_time, bar_dist, miss_text in zip(bars_time, bars_dist, miss_texts):
                if miss_text is not None:
                    for bar in (bar_time, bar_dist):
                        bar.set_hatch('//')
                        bar.set_alpha(0.45)
                    ax_dist.annotate(miss_text, (bar_dist.get_x() + bar_dist.get_width() / 2, bar_dist.get_height()),
                                     xytext=(0, 3), textcoords='offset points', rotation=90, ha='center', va='bottom', fontsize=7)
        ax_time.axhline(0.0, color='k', lw=0.8)
        ax_time.set_xticks(number)
        ax_time.set_xticklabels([f'WP{i}' for i in number])
        ax_time.set_ylabel('time to settle after the reference arrived (s)')
        ax_time.set_title(f'{path_name}: time to settle at each waypoint', fontsize=10)
        ax_time.grid(True, axis='y', linestyle=':', alpha=0.6)
        handles, labels = ax_time.get_legend_handles_labels()
        handles.append(Patch(facecolor='lightgrey', hatch='//', label='hatched: not attained'))
        ax_time.legend(handles=handles, fontsize='small', ncol=2)
        ax_dist.axhline(100 * ARRIVE_THRESHOLD_POS, color='g', ls='--', lw=1.2, label=f'settle radius ({100 * ARRIVE_THRESHOLD_POS:.0f} cm)')
        ax_dist.set_ylim(0.0, 2.2 * largest_distance)        # room for the labels
        ax_dist.set_xticks(number)
        ax_dist.set_xticklabels([f'WP{i}' for i in number])
        ax_dist.set_ylabel('distance from the waypoint (cm)')
        ax_dist.set_title(f'{path_name}: where each got to, when the path moved on or the run ended', fontsize=10)
        ax_dist.grid(True, axis='y', linestyle=':', alpha=0.6)
        ax_dist.legend(fontsize='small', ncol=2)
    for n in range(len(waypoint_paths), rows * columns):
        for r in range(2):
            axs[2 * (n // columns) + r, n % columns].axis('off')
    plt.tight_layout()
    plt.show()


def plot_marker_times(results, paths, controllers, columns=3):
    """
    Henry's waypoint-executed eights ("who gets there first"): for each marker, how long after its scheduled time
    (the continuous eight's) each controller got within reach of it (ARRIVE_THRESHOLD_POS), in pass order. Gaps: not
    reached by the end of the run. A controller that keeps up stays near zero; one that falls behind climbs.
    """
    marker_paths = [name for name in paths if 'markers' in paths[name]]
    if not marker_paths:
        return
    rows = int(np.ceil(len(marker_paths) / columns))
    fig, axs = plt.subplots(rows, columns, figsize=(6 * columns, 4 * rows), squeeze=False)
    for ax, path_name in zip(axs.ravel(), marker_paths):
        path = paths[path_name]
        passes = np.arange(1, len(path['markers']) + 1)
        scheduled = path['t'][path['marker_steps']] + DT_CTRL
        for controller in controllers:
            reached = results[(path_name, controller)]['marker_reached']
            behind = np.array([np.nan if r is None else r - s for r, s in zip(reached, scheduled)])
            n_reached = int(np.sum(~np.isnan(behind)))
            ax.plot(passes, behind, 'o-', color=CONTROLLERS[controller][1], lw=1.4, ms=4,
                    label=f'{controller}: {n_reached}/{len(passes)}')
        for lap_end in range(path['markers_per_lap'], len(passes), path['markers_per_lap']):
            ax.axvline(lap_end + 0.5, color='k', lw=0.8, alpha=0.4)
        ax.set_xticks(passes)
        ax.set_xticklabels([f'M{(n - 1) % path["markers_per_lap"] + 1}' for n in passes], fontsize=7)
        ax.set_ylabel('reached after its scheduled time (s)')
        ax.set_title(f'{path_name}: when each reached each marker (legend: markers reached)', fontsize=10)
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(fontsize='small', ncol=2)
    for ax in axs.ravel()[len(marker_paths):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def plot_marker_misses(results, paths, controllers, columns=3):
    """
    For each path with markers (the figure eights): how far each controller was from each marker as the reference
    passed it, in pass order (M1 to M7 on each lap). A controller that lags or cuts in is far from the markers.
    """
    marker_paths = [name for name in paths if 'markers' in paths[name]]
    if not marker_paths:
        return
    rows = int(np.ceil(len(marker_paths) / columns))
    fig, axs = plt.subplots(rows, columns, figsize=(6 * columns, 4 * rows), squeeze=False)
    for ax, path_name in zip(axs.ravel(), marker_paths):
        path = paths[path_name]
        passes = np.arange(1, len(path['markers']) + 1)
        for controller in controllers:
            result = results[(path_name, controller)]
            miss = np.full(len(passes), np.nan)       # gaps: the reference never got there
            for i, step in enumerate(result['marker_steps']):
                if step is not None:
                    miss[i] = 100 * np.hypot(*(result['pos'][step, :2] - path['markers'][i, :2]))
            ax.plot(passes, miss, 'o-', color=CONTROLLERS[controller][1], lw=1.4, ms=4, label=controller)
        for lap_end in range(path['markers_per_lap'], len(passes), path['markers_per_lap']):
            ax.axvline(lap_end + 0.5, color='k', lw=0.8, alpha=0.4)
        ax.set_xticks(passes)
        ax.set_xticklabels([f'M{(n - 1) % path["markers_per_lap"] + 1}' for n in passes], fontsize=7)
        ax.set_ylabel('distance from the marker (cm)')
        ax.set_title(f'{path_name}: as the reference passed each marker', fontsize=10)
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(fontsize='small', ncol=2)
    for ax in axs.ravel()[len(marker_paths):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def figure_eight_path(size, period, laps=2, ramp=0.3, swing=0.3, dt=0.02):
    """Figure eight in Dana's path format (t, pos, vel; global frame): size [m], period per lap [s]."""
    t = np.arange(0.0, laps * period + 1.0, dt)
    rate = 2 * np.pi / period
    phase = rate * (t - ramp * (1 - np.exp(-t / ramp)))
    phase_dot = rate * (1 - np.exp(-t / ramp))
    pos = np.column_stack((size * np.sin(phase), 0.5 * size * np.sin(2 * phase), swing * np.sin(phase)))
    vel = np.column_stack((size * np.cos(phase) * phase_dot, size * np.cos(2 * phase) * phase_dot, swing * np.cos(phase) * phase_dot))
    path = {'t': t, 'pos': pos, 'vel': vel}
    return path


def largest_distance_off_path(robot_xy, reference_xy):
    """Largest distance from the robot's positions to the reference path (the line through its points), ignoring timing."""
    a = reference_xy[:-1]
    ab = reference_xy[1:] - a
    ab_len2 = np.maximum(np.sum(ab**2, axis=1), 1e-12)
    distances = np.zeros(len(robot_xy))
    for i, p in enumerate(robot_xy):
        s = np.clip(np.sum((p - a) * ab, axis=1) / ab_len2, 0.0, 1.0)
        distances[i] = np.min(np.hypot(*(a + s[:, None] * ab - p).T))
    return distances.max()


def heading_arrows(ax, xy, theta, color, length, count=12, alpha=0.8):
    """Arrows along a path showing which way the robot faces: forward is +y of the robot, so the direction is
    R(theta) [0, 1] = (-sin theta, cos theta) (as in Dana's plots). count arrows, evenly spaced in time."""
    every = max(1, len(xy) // count)
    idx = np.arange(0, len(xy), every)
    ax.quiver(xy[idx, 0], xy[idx, 1], -np.sin(theta[idx]) * length, np.cos(theta[idx]) * length,
              color=color, angles='xy', scale_units='xy', scale=1.0, width=0.004, alpha=alpha)


def plot_paths(results, paths, controllers, columns=3):
    """X-Y paths of every controller, one subplot per path, with heading arrows (grey: the reference's heading)."""
    rows = int(np.ceil(len(paths) / columns))
    fig, axs = plt.subplots(rows, columns, figsize=(6 * columns, 6 * rows), squeeze=False)
    for ax, path_name in zip(axs.ravel(), paths):
        ref = results[(path_name, controllers[0])]['pos_ref']
        size = max(np.ptp(ref[:, 0]), np.ptp(ref[:, 1]), 0.2)       # arrow length: 8% of the path's size
        ax.plot(ref[:, 0], ref[:, 1], 'k:', lw=1.8, label='Reference')
        heading_arrows(ax, ref[:, :2], ref[:, 2], 'grey', 0.08 * size, alpha=0.6)
        draw_waypoints(ax, paths[path_name], 0.08 * size)     # MERGED: Dana's waypoint marks
        draw_arrival_positions(ax, results, path_name, paths[path_name], controllers)     # MERGED: where each was on time
        if 'waypoints' in paths[path_name]:     # MERGED: an X for each waypoint not attained, where it was at the end
            drew = [draw_not_attained(ax, results[(path_name, c)], paths[path_name]['waypoints'], CONTROLLERS[c][1]) for c in controllers]
            if any(drew):
                ax.plot([], [], 'X', color='grey', mec='k', ms=10, ls='none', label='X: not attained (where it was when the path moved on)')
        for controller in controllers:
            pos = results[(path_name, controller)]['pos']
            ax.plot(pos[:, 0], pos[:, 1], color=CONTROLLERS[controller][1], lw=2.0, label=controller)
            heading_arrows(ax, pos[:, :2], pos[:, 2], CONTROLLERS[controller][1], 0.08 * size)
        ax.set_aspect('equal', 'datalim'); ax.grid(True, linestyle=':', alpha=0.6)
        ax.set_title(path_name); ax.set_xlabel('Global X (m)'); ax.set_ylabel('Global Y (m)'); ax.legend(loc='best', fontsize='small')
    for ax in axs.ravel()[len(paths):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def plot_errors(results, paths, controllers, columns=3):
    """For each path: Dana's position error over time (top), the perpendicular error (the shortest distance to the
    track, time ignored; middle) and the heading error (bottom)."""
    rows = int(np.ceil(len(paths) / columns))
    fig, axs = plt.subplots(3 * rows, columns, figsize=(6 * columns, 10 * rows), squeeze=False)
    for n, path_name in enumerate(paths):
        ax_pos = axs[3 * (n // columns), n % columns]
        ax_perp = axs[3 * (n // columns) + 1, n % columns]
        ax_head = axs[3 * (n // columns) + 2, n % columns]
        mark_settled_times([ax_pos, ax_perp, ax_head], results, path_name, controllers)     # MERGED: when each settled at each waypoint
        for controller in controllers:
            res = results[(path_name, controller)]
            perpendicular, lag = path_errors(res['pos'][:, :2], res['pos_ref'][:, :2])
            ax_pos.plot(res['t'], 100 * res['dist_err'], color=CONTROLLERS[controller][1], lw=1.6, label=controller)
            ax_perp.plot(res['t'], 100 * perpendicular, color=CONTROLLERS[controller][1], lw=1.6, label=controller)
            ax_head.plot(res['t'], np.degrees(res['th_err']), color=CONTROLLERS[controller][1], lw=1.6, label=controller)
        ax_pos.grid(True, linestyle=':', alpha=0.6); ax_pos.set_title(path_name)
        ax_pos.set_ylabel('Position error (cm)'); ax_pos.legend(loc='best', fontsize='small')
        ax_perp.grid(True, linestyle=':', alpha=0.6); ax_perp.set_ylabel('Perpendicular error (cm)')
        ax_head.grid(True, linestyle=':', alpha=0.6)
        ax_head.set_xlabel('Time (s)'); ax_head.set_ylabel('Heading error (deg)')
    for n in range(len(paths), rows * columns):
        for r in range(3):
            axs[3 * (n // columns) + r, n % columns].axis('off')
    plt.tight_layout()
    plt.show()


def plot_effort(results, paths, controllers, columns=3):
    """How hard each controller drives the motors, from the 1 ms logs. For each path, over time:
      1. the largest wheel voltage, max |V_i|;
      2. the total motor current, sum |I_i|;
      3. the electrical power, sum |V_i I_i|, with each controller's total energy (J) in the legend.
    A controller that tracks better by spending more shows up here."""
    rows = int(np.ceil(len(paths) / columns))
    fig, axs = plt.subplots(3 * rows, columns, figsize=(6 * columns, 10 * rows), squeeze=False)
    for n, path_name in enumerate(paths):
        ax_v = axs[3 * (n // columns), n % columns]
        ax_i = axs[3 * (n // columns) + 1, n % columns]
        ax_p = axs[3 * (n // columns) + 2, n % columns]
        mark_settled_times([ax_v, ax_i, ax_p], results, path_name, controllers)     # MERGED: when each settled at each waypoint
        for controller in controllers:
            res = results[(path_name, controller)]
            t = res['t_1ms']
            power = np.sum(np.abs(res['V_1ms'] * res['I_1ms']), axis=1)
            energy = np.sum(power) * (t[1] - t[0])
            color = CONTROLLERS[controller][1]
            ax_v.plot(t, np.max(np.abs(res['V_1ms']), axis=1), color=color, lw=1.2, label=controller)
            ax_i.plot(t, np.sum(np.abs(res['I_1ms']), axis=1), color=color, lw=1.2, label=controller)
            ax_p.plot(t, power, color=color, lw=1.2, label=f"{controller}: {energy:.1f} J")
        ax_v.set_title(path_name); ax_v.set_ylabel('Largest wheel voltage (V)'); ax_v.legend(loc='best', fontsize='small')
        ax_i.set_ylabel('Total motor current (A)')
        ax_p.set_ylabel('Electrical power (W)'); ax_p.set_xlabel('Time (s)'); ax_p.legend(loc='best', fontsize='small', title='energy')
        for ax in (ax_v, ax_i, ax_p):
            ax.grid(True, linestyle=':', alpha=0.6)
    for n in range(len(paths), rows * columns):
        for r in range(3):
            axs[3 * (n // columns) + r, n % columns].axis('off')
    plt.tight_layout()
    plt.show()



## 1. Dana's waypoint path

Four waypoints, 5 s segments, a 0.5 s hold at each, with her controller list and her six plots. Each segment leaves its waypoint at exactly 90 degrees to the direction it arrived from (`PathGeneratorCurvedWithHold`), so cutting corners or lagging shows clearly. Every segment curves like an arc (it arrives at the mirror of the angle it left at); the first leaves 45 degrees off its straight line. Dana's fixed time: 0.5 s hold at each waypoint; the table gives each waypoint's settled time (X: not settled within its hold). The same path, waypoint-executed, is in section 5.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.patches import Circle
import time

# =======================================================================
# SETUP MULTI-WAYPOINT TRAJECTORY WITH HOLDS
# =======================================================================
PathGenerator_Multi = PathGeneratorWithHold(dt=DT_CTRL)



WAYPOINTS = [
    np.array([0.0, 0.0, 0.0]),
    np.array([2.2, 1.2, 0.50]),
    np.array([1.0, 3.5, -0.60]),
    np.array([3.2, 2.8, 0.785])
]
x_wp = 0.5*np.cos(np.radians(-90))
y_wp = 0.5*np.sin(np.radians(-90))
t_wp=np.radians(-180)
#WAYPOINTS = [    np.array([0.0, 0.0, 0.0]),    np.array([x_wp, y_wp, t_wp])]

SEGMENT_DURATION = 5.0
HOLD_DURATION = 0.5

# MERGED (11-13): right-angle corners at her waypoints (each segment leaves at 90 degrees to its arrival), so cutting corners or lagging shows clearly
path_data=PathGeneratorCurvedWithHold(dt=DT_CTRL).generate_waypoints_sequence(waypoints=WAYPOINTS,segment_duration=SEGMENT_DURATION,segment_hold=HOLD_DURATION)

WP_TIMESTAMPS = path_data['segment_boundaries']
POS_REF = path_data['pos']
VEL_REF = path_data['vel']
N_STEPS = len(path_data['pos'])
T_REF = path_data['t']
V_LOCAL_REF = path_data['pos_local']


print(f"=======================================================================")
print(f"   MULTI-WAYPOINT TRAJECTORY GENERATED ({N_STEPS} steps | {T_REF[-1]:.2f} s)")
print(f"=======================================================================")
for idx, wp in enumerate(WAYPOINTS):
    print(f" • Waypoint {idx}: [{wp[0]:.2f}m, {wp[1]:.2f}m, {np.degrees(wp[2]):.1f}°] | Hold ends: {WP_TIMESTAMPS[idx]:.2f} s")
print(f"=======================================================================")


In [ ]:
# =======================================================================
# EXECUTE WAYPOINT SIMULATION RUNNER (Dana's list, plus the tuned PID)
# =======================================================================
WAYPOINT_CONTROLLERS = ['PID tuned', 'Simple MPC', 'LQR', 'DOB MPC', 'Coulomb MPC', 'Linearized MPC']
Controller_Dictionary = {name: (name, None, CONTROLLERS[name][1]) for name in WAYPOINT_CONTROLLERS}    # Dana's plots read the colours here

print(f"\nExecuting Multi-Waypoint Benchmark Simulations .........")
results_multi_db = {}
for name in WAYPOINT_CONTROLLERS:
    print(f"Running {name} Simulation")
    results_multi_db[name] = run_simulation(name, path_data)
print(f"\n Completed all simulations")


In [ ]:
print("\n" + "="*164)
print("              WAYPOINT PATH TRACKING SUMMARY")
print("="*164)
# print headers for table
headers = ["Controller", "RMS Time Err (cm)", "Max Spatial Err (cm)", "Max Slip (mm/s)", "Step Time (ms)", "Max Input (V) "]
for i in range(len(WAYPOINTS)-1):
    headers.append(f"WP{i+1} Settled (s)")    
for header in headers:
    print(f"{header:<17} | ",end="")
print("")
print("-" * 164)

# print content per controller
for name in Controller_Dictionary.keys():
    res = results_multi_db[name]
    rms_time_err=np.sqrt(np.mean(res['pos_err']**2)) * 100.0
    
    # Orthogonal cross-track spatial path distance error
    dx = res['pos'][:, 0, None] - POS_REF[:, 0][None, :]
    dy = res['pos'][:, 1, None] - POS_REF[:, 1][None, :]
    spatial_path_dist_max =np.max( np.min(np.hypot(dx, dy), axis=1))*100
    slip_max = np.max(np.abs(res['v_slip'])) * 1000.0 # mm/s
    print(f"{name:<17} | {rms_time_err:<17.2f} | {spatial_path_dist_max:<17.2f} | {slip_max:<17.2f} | {(res['exec_time'] / len(res['t'])) * 1000.0 :<17.2f} | {np.max(np.abs(res['V_cmd'])):<17.2f} |",end="")
    for arrival in res['arrival_times'][1:]:
        arrival_time = f"{arrival:.2f}s" if arrival else 'X not attained'
        print(f"{arrival_time:<17} | ",end="")
    print("")

print("="*164)

# =======================================================================
# DIAGNOSTIC VISUALIZATION SUITE (6 PLOTS)
# =======================================================================

# PLOT 1: 2D Multi-Waypoint Executed Trajectories & Target Spheres
fig1, ax1 = plt.subplots(figsize=(11, 8))
ax1.plot(POS_REF[:, 0], POS_REF[:, 1], 'k--', lw=2.0, alpha=0.6, label='Desired Reference Path')

for idx, wp in enumerate(WAYPOINTS):
    ax1.plot(wp[0], wp[1], 'ro', ms=8, zorder=8)
    circle = Circle((wp[0], wp[1]), ARRIVE_THRESHOLD_POS, color='g', fill=True, alpha=0.25, zorder=3)
    ax1.add_patch(circle)
    ax1.text(wp[0]+0.08, wp[1]+0.08, f'WP{idx}\n({wp[0]:.1f}m, {wp[1]:.1f}m)', fontsize=9.5, fontweight='bold', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.85))
    u_fwd = -np.sin(wp[2]) * 0.2
    v_fwd =  np.cos(wp[2]) * 0.2
    ax1.annotate('', xy=(wp[0] + u_fwd, wp[1] + v_fwd), xytext=(wp[0], wp[1]),
                        arrowprops=dict(arrowstyle="->", color='g', lw=1))
    
for name, ctuple in Controller_Dictionary.items():
    color = ctuple[2]
    res = results_multi_db[name]
    ax1.plot(res['pos'][:, 0], res['pos'][:, 1], color=color, lw=1.8, label=name, zorder=5)
    pos=res['pos']
    step_jump = max(1, len(res['pos']) // 10)
    indices = np.arange(0, len(res['pos']), step_jump)
    for idx in indices:
        xi, yi, thi = res['pos'][idx, 0], res['pos'][idx, 1], pos[idx, 2]
        u_fwd = -np.sin(thi) * 0.2
        v_fwd =  np.cos(thi) * 0.2
        ax1.annotate('', xy=(xi + u_fwd, yi + v_fwd), xytext=(xi, yi),
                    arrowprops=dict(arrowstyle="->", color=color, lw=1))

# MERGED: an X for each waypoint a controller did not attain, where it was at the end (joined to that waypoint)
drew = [draw_not_attained(ax1, results_multi_db[name], WAYPOINTS, ctuple[2]) for name, ctuple in Controller_Dictionary.items()]
if any(drew):
    ax1.plot([], [], 'X', color='grey', mec='k', ms=10, ls='none', label='X: not attained (where it was when the path moved on)')
ax1.set_title(f'Waypoint Executed 2D Controller Trajectories', fontsize=12, fontweight='bold')
ax1.set_xlabel('Global X Position (m)', fontsize=11)
ax1.set_ylabel('Global Y Position (m)', fontsize=11)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(loc='upper left', fontsize='small')
ax1.set_aspect('equal')
plt.tight_layout()
plt.show()


# PLOT 2: Robot Linear Speed Magnitude & Radial Angular Velocity Subplots Over Time
fig3, axs3 = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

v_lin_ref = np.hypot(VEL_REF[:, 0], VEL_REF[:, 1])
w_yaw_ref_deg = np.degrees(np.abs(VEL_REF[:, 2]))

# Subplot 3A: Linear Speed Magnitude ||v_lin||
# MERGED: each controller has its own timeline (the path waits at each waypoint until it settles): its own reference, dashed
for name, ctuple in Controller_Dictionary.items():
    color = ctuple[2]
    res_v = results_multi_db[name]['v_local']
    v_lin_act = np.hypot(res_v[:, 0], res_v[:, 1])
    vel_ref_own = results_multi_db[name]['vel_ref']
    axs3[0].plot(results_multi_db[name]['t'], np.hypot(vel_ref_own[:, 0], vel_ref_own[:, 1]), color=color, ls='--', lw=1.0, alpha=0.6, zorder=2)
    axs3[0].plot(results_multi_db[name]['t'], v_lin_act, color=color, lw=1.8, label=name, zorder=4)
axs3[0].set_ylabel('Linear Speed (m/s)', fontsize=10, fontweight='bold')
axs3[0].set_title('Chassis Linear Speed Magnitude ||v_lin|| = sqrt(vx^2 + vy^2)', fontsize=11, fontweight='bold')
axs3[0].grid(True, linestyle=':', alpha=0.6)
axs3[0].legend(loc='upper right', fontsize='small', ncol=3)

# Subplot 3B: Radial / Angular Velocity Magnitude |w_yaw|
# MERGED: each controller's own reference, dashed
for name, ctuple in Controller_Dictionary.items():
    color = ctuple[2]
    w_yaw_act_deg = np.degrees(np.abs(results_multi_db[name]['v_local'][:, 2]))
    axs3[1].plot(results_multi_db[name]['t'], np.degrees(np.abs(results_multi_db[name]['vel_ref'][:, 2])), color=color, ls='--', lw=1.0, alpha=0.6, zorder=2)
    axs3[1].plot(results_multi_db[name]['t'], w_yaw_act_deg, color=color, lw=1.8, label=name, zorder=4)
axs3[1].set_ylabel('Angular Rate (°/s)', fontsize=10, fontweight='bold')
axs3[1].set_xlabel('Simulation Time (s)', fontsize=11)
axs3[1].set_title('Chassis Radial/Angular Velocity Magnitude |w_yaw| ', fontsize=11, fontweight='bold')
axs3[1].grid(True, linestyle=':', alpha=0.6)
axs3[1].legend(loc='upper right', fontsize='small', ncol=3)

plt.tight_layout()
plt.show()

# PLOT 4: Relative Tracking Errors Delta X, Delta Y, Delta Theta Subplots
fig4, axs4 = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
x_ref = POS_REF[:, 0]
y_ref = POS_REF[:, 1]
th_ref = np.degrees(POS_REF[:, 2])

# Subplot A: Delta X Error
# MERGED: when each controller settled at each waypoint, dotted in its colour (each has its own timeline)
for name in WAYPOINT_CONTROLLERS:
    for settled_time in results_multi_db[name]['settled']:
        if settled_time is not None:
            axs4[0].axvline(settled_time, color=CONTROLLERS[name][1], linestyle=':', alpha=0.8)


for name, ctuple in Controller_Dictionary.items():
    color = ctuple[2]
    dx_cm = (results_multi_db[name]['pos'][:, 0] - results_multi_db[name]['pos_ref'][:, 0]) * 100.0   # MERGED: its own reference
    axs4[0].plot(results_multi_db[name]['t'], dx_cm, color=color, lw=1.8, label=name)
axs4[0].axhline(ARRIVE_THRESHOLD_POS * 100.0, color='g', linestyle=':', alpha=0.7, label=f'±{ARRIVE_THRESHOLD_POS * 100.0:.1f} cm Target Deadband Bounds')
axs4[0].axhline(-ARRIVE_THRESHOLD_POS * 100.0, color='g', linestyle=':', alpha=0.7)
axs4[0].set_ylabel('Delta X Error (cm)', fontsize=10, fontweight='bold')
axs4[0].set_title('Relative Position Tracking Error: Delta X = X_actual - X_ref (cm)', fontsize=11, fontweight='bold')
axs4[0].grid(True, linestyle=':', alpha=0.6)
axs4[0].legend(loc='upper right', fontsize='small', ncol=3)

# Subplot B: Delta Y Error
# MERGED: when each controller settled at each waypoint, dotted in its colour (each has its own timeline)
for name in WAYPOINT_CONTROLLERS:
    for settled_time in results_multi_db[name]['settled']:
        if settled_time is not None:
            axs4[1].axvline(settled_time, color=CONTROLLERS[name][1], linestyle=':', alpha=0.8)

for name, ctuple in Controller_Dictionary.items():
    color = ctuple[2]
    dy_cm = (results_multi_db[name]['pos'][:, 1] - results_multi_db[name]['pos_ref'][:, 1]) * 100.0   # MERGED: its own reference
    axs4[1].plot(results_multi_db[name]['t'], dy_cm, color=color, lw=1.8, label=name)
axs4[1].axhline(ARRIVE_THRESHOLD_POS * 100.0, color='g', linestyle=':', alpha=0.7, label=f'±{ARRIVE_THRESHOLD_POS * 100.0:.1f} cm Target Deadband Bounds')
axs4[1].axhline(-ARRIVE_THRESHOLD_POS * 100.0, color='g', linestyle=':', alpha=0.7)
axs4[1].set_ylabel('Delta Y Error (cm)', fontsize=10, fontweight='bold')
axs4[1].set_title('Relative Position Tracking Error: Delta Y = Y_actual - Y_ref (cm)', fontsize=11, fontweight='bold')
axs4[1].grid(True, linestyle=':', alpha=0.6)
axs4[1].legend(loc='upper right', fontsize='small', ncol=3)

# Subplot C: Delta Theta Error
# MERGED: when each controller settled at each waypoint, dotted in its colour (each has its own timeline)
for name in WAYPOINT_CONTROLLERS:
    for settled_time in results_multi_db[name]['settled']:
        if settled_time is not None:
            axs4[2].axvline(settled_time, color=CONTROLLERS[name][1], linestyle=':', alpha=0.8)

for name, ctuple in Controller_Dictionary.items():
    color = ctuple[2]
    dth_deg = np.degrees((results_multi_db[name]['pos'][:, 2] - results_multi_db[name]['pos_ref'][:, 2] + np.pi) % (2.0 * np.pi) - np.pi)
    axs4[2].plot(results_multi_db[name]['t'], dth_deg, color=color, lw=1.8, label=name)
axs4[2].axhline(ARRIVE_THRESHOLD_THETA , color='g', linestyle=':', alpha=0.7, label=f'±{ARRIVE_THRESHOLD_THETA:.1f} cm Target Theta')
axs4[2].axhline(-ARRIVE_THRESHOLD_THETA , color='g', linestyle=':', alpha=0.7)
axs4[2].set_ylabel('Delta Theta Error (°)', fontsize=10, fontweight='bold')
axs4[2].set_xlabel('Simulation Time (s)', fontsize=11)
axs4[2].set_title('Relative Heading Angle Tracking Error: Delta Theta = Theta_actual - Theta_ref (°)', fontsize=11, fontweight='bold')
axs4[2].grid(True, linestyle=':', alpha=0.6)
axs4[2].legend(loc='upper right', fontsize='small', ncol=3)

plt.tight_layout()
plt.show()


# PLOT 6: Motor Terminal Voltages & Armature Currents Subplots (Twin Y-Axes per Controller Type)
fig6, axs6 = plt.subplots(len(Controller_Dictionary), 1, figsize=(11, 15), sharex=True)

v_lim = SIMULATION_PARAMATERS['V_max']
i_lim = SIMULATION_PARAMATERS['I_max']

for idx, (name, ctuple) in enumerate(Controller_Dictionary.items()):
    ax_v = axs6[idx]
    ax_i = ax_v.twinx()
    
    res = results_multi_db[name]
    V_data = res['V_cmd']
    I_data = res['I']
    
    # Motor Voltages (Primary Left Axis)
    colors_v = ['blue', 'cyan', 'navy', 'royalblue']
    for w in range(4):
        ax_v.plot(res['t'], V_data[:, w], color=colors_v[w], lw=1.2, alpha=0.85, label=f'V{w+1}')
    
    # Supply Voltage Limits (±V_max)
    ax_v.axhline(v_lim, color='red', linestyle='--', lw=1.2, alpha=0.75, label=f'±{v_lim:.1f}V Voltage Ceiling')
    ax_v.axhline(-v_lim, color='red', linestyle='--', lw=1.2, alpha=0.75)
    ax_v.set_ylim(-v_lim * 1.15, v_lim * 1.15)
    ax_v.set_ylabel('Voltage (V)', fontsize=9.5, fontweight='bold', color='darkblue')
    
    # Motor Currents (Secondary Right Axis)
    colors_i = ['darkred', 'crimson', 'darkorange', 'coral']
    for w in range(4):
        ax_i.plot(res['t'], I_data[:, w], color=colors_i[w], linestyle='--', lw=1.2, alpha=0.85, label=f'I{w+1}')
        
    # Over-Current Limits (±I_max)
    ax_i.axhline(i_lim, color='darkorange', linestyle=':', lw=1.5, alpha=0.85, label=f'±{i_lim:.1f}A Over-Current Limit')
    ax_i.axhline(-i_lim, color='darkorange', linestyle=':', lw=1.5, alpha=0.85)
    ax_i.set_ylim(-i_lim * 1.2, i_lim * 1.2)
    ax_i.set_ylabel('Current (A)', fontsize=9.5, fontweight='bold', color='darkred')
    
    # Labels & Title
    ax_v.set_title(f'{name} — Motor Terminal Voltages (V) & Armature Currents (A) Timeline', fontsize=11, fontweight='bold')
    ax_v.grid(True, linestyle=':', alpha=0.5)
    
    # Combine Legends
    lines_v, labels_v = ax_v.get_legend_handles_labels()
    lines_i, labels_i = ax_i.get_legend_handles_labels()
    ax_v.legend(lines_v + lines_i, labels_v + labels_i, loc='upper right', fontsize=8, ncol=5)

axs6[-1].set_xlabel('Simulation Time (s)', fontsize=11)
plt.tight_layout()
plt.show()

# PLOT 7 : Wheel Contact Patch Slip Velocity Subplots per Controller Type
fig7, axs7 = plt.subplots(len(Controller_Dictionary), 1, figsize=(11, 3 * len(Controller_Dictionary)), sharex=True)

wheel_colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:purple']
vs_mm_s = SIMULATION_PARAMATERS['slip_band'] * 1000.0 # MERGED: the plant's slip band (no damping parameter)

for idx, (name, ctuple) in enumerate(Controller_Dictionary.items()):
    ax_s = axs7[idx]
    res = results_multi_db[name]
    v_slip_mm_s = res['v_slip'] * 1000.0 # Convert m/s to mm/s
    
    for w in range(4):
        ax_s.plot(res['t'], v_slip_mm_s[:, w], color=wheel_colors[w], lw=1.4, alpha=0.85, label=f'Wheel {w+1} v_slip,{w+1}')
        
    ax_s.axhline(vs_mm_s, color='black', linestyle='--', lw=1.2, alpha=0.75, label=f'±{vs_mm_s:.1f} mm/s Slip Band')
    ax_s.axhline(-vs_mm_s, color='black', linestyle='--', lw=1.2, alpha=0.75)
    
    ax_s.set_ylabel('v_slip (mm/s)', fontsize=9.5, fontweight='bold')
    ax_s.set_title(f'{name} — Wheel Contact Patch Slip Velocities v_slip,i (mm/s) Timeline', fontsize=11, fontweight='bold')
    ax_s.grid(True, linestyle=':', alpha=0.5)
    ax_s.legend(loc='upper right', fontsize=8, ncol=5)

axs7[-1].set_xlabel('Simulation Time (s)', fontsize=11)
plt.tight_layout()
plt.show()



In [ ]:
# Where each controller was when the reference reached each waypoint, and when it settled there
dana_paths = {"Dana's waypoints (right-angle corners)": path_data}
dana_results = {("Dana's waypoints (right-angle corners)", name): results_multi_db[name] for name in WAYPOINT_CONTROLLERS}
plot_paths(dana_results, dana_paths, WAYPOINT_CONTROLLERS, columns=1)
plot_waypoint_timing(dana_results, dana_paths, WAYPOINT_CONTROLLERS, columns=1)
plot_waypoint_closeups(dana_results, dana_paths, WAYPOINT_CONTROLLERS)


## 2. Dana's benchmark

Her first path: (0, 0, 0) to (3, 3, π/4) in 8 s and in 4 s, now with a hold at the end. The 4 s path needs more wheel speed than 12 V gives, so every controller falls behind on it.


In [ ]:
path_generator = PathGeneratorWithHold(dt=DT_CTRL)
benchmark_paths = {
    "Dana 8 s": path_generator.generate_waypoints_sequence([np.zeros(3), np.array([3.0, 3.0, np.pi / 4])], segment_duration=8.0, segment_hold=HOLD_DURATION),
    "Dana 4 s (too fast)": path_generator.generate_waypoints_sequence([np.zeros(3), np.array([3.0, 3.0, np.pi / 4])], segment_duration=4.0, segment_hold=HOLD_DURATION),
}
benchmark_controllers = list(WAYPOINT_CONTROLLERS)     # the same controllers in every test
benchmark_results = run_and_report("Dana's benchmark", benchmark_paths, benchmark_controllers)
plot_paths(benchmark_results, benchmark_paths, benchmark_controllers, columns=2)
plot_errors(benchmark_results, benchmark_paths, benchmark_controllers, columns=2)
plot_effort(benchmark_results, benchmark_paths, benchmark_controllers, columns=2)
plot_waypoint_timing(benchmark_results, benchmark_paths, benchmark_controllers, columns=2)
plot_waypoint_closeups(benchmark_results, benchmark_paths, benchmark_controllers)


## 3. Test motions

Henry's attainable paths, from Dana's generator with a hold at each waypoint: her arena (random waypoints, 10 s segments) and her quadrant waypoints (12 s segments), quick hops in a ±0.5 m area (1.25 s segments), and Henry's continuous figure eights (two laps; the lap time in brackets). On the eights, 7 markers per lap equally spaced along the eight are a visual check only (the path does not stop or wait at them): the plots show where each controller was as the reference passed each marker, and how far off it was.

The waypoint paths are waypoint-executed: the path waits at each waypoint until that controller has settled, so each controller has its own timeline.

The peak wheel speed each path needs is a path check, in `ModelChecks.ipynb`. For this motor (Pololu) the limit is top speed, not grip: the 0.4 m eight at 3.4 s per lap needs about 9.3 V of back-EMF of the 12 V (inside the limit); at 2.5 s it needs 12.6 V (past it, so it should fall behind on the fast parts). The 0.3 m eight at 2.2 s is the tight one (about 5 m/s²).

"Perp" and "Lag" split the error into the distance from the track and the distance behind along it (RMS). "Settled at" gives the time it settled at each waypoint, on its own timeline; on the eights, how far it was from the markers as the reference passed them (mean and worst).


In [ ]:
arena_half = 9.0 / 2.0
arena_margin = 0.5
workspace_9m = {'x_min': -arena_half + arena_margin, 'x_max': arena_half - arena_margin,
                'y_min': -arena_half + arena_margin, 'y_max': arena_half - arena_margin, 'theta_min': -np.pi, 'theta_max': np.pi}
quadrant_waypoints = [np.array([0.0, 0.0, 0.0]), np.array([3.2, 2.8, np.radians(90.0)]), np.array([-2.5, 3.5, np.radians(180.0)]),
                      np.array([-3.5, -2.8, np.radians(-90.0)]), np.array([2.8, -3.2, 0.0])]
small_area = {'x_min': -0.5, 'x_max': 0.5, 'y_min': -0.5, 'y_max': 0.5, 'theta_min': -np.pi / 2, 'theta_max': np.pi / 2}

motion_paths = {
    "Random (10 s segments)": path_generator.generate_random_sequence(num_segments=4, workspace_bounds=workspace_9m, segment_duration=10.0, segment_hold=HOLD_DURATION, random_seed=42),
    "Quadrants (12 s segments)": path_generator.generate_waypoints_sequence(quadrant_waypoints, segment_duration=12.0, segment_hold=HOLD_DURATION),
    "Quick hops (1.25 s)": path_generator.generate_random_sequence(num_segments=6, workspace_bounds=small_area, segment_duration=1.25, segment_hold=HOLD_DURATION, random_seed=7),
    "Figure eight 0.4 m (3.4 s)": add_eight_markers(figure_eight_path(0.4, 3.4), 0.4, markers_per_lap=7),
    "Figure eight 0.4 m (2.5 s)": add_eight_markers(figure_eight_path(0.4, 2.5), 0.4, markers_per_lap=7),
    "Figure eight 0.3 m (2.2 s)": add_eight_markers(figure_eight_path(0.3, 2.2), 0.3, markers_per_lap=7),
}
motion_controllers = list(WAYPOINT_CONTROLLERS)     # the same controllers in every test
motion_results = run_and_report("Test motions", motion_paths, motion_controllers)
plot_paths(motion_results, motion_paths, motion_controllers)
plot_errors(motion_results, motion_paths, motion_controllers)
plot_effort(motion_results, motion_paths, motion_controllers)
plot_waypoint_timing(motion_results, motion_paths, motion_controllers)
plot_waypoint_closeups(motion_results, motion_paths, motion_controllers)
plot_marker_misses(motion_results, motion_paths, motion_controllers)


## 4. Dana's turning test

Move from (0, 0, 0) to (3, 3, π/2) while turning, and shorten the time; also a half turn, (3, 3, π). Each ends with a hold.

Shorter times need more wheel speed (the turn adds to it). The peak wheel speed each case needs, as a share of the top speed at 12 V (about 47 rad/s; from the path check in `ModelChecks.ipynb`). Above 100% the robot cannot keep up:

| Move | 8 s | 5 s | 4 s |
|---|---|---|---|
| to (3, 3, π/2) | 59% | 95% | 118% (too fast) |
| to (3, 3, π) | 75% | 120% (too fast) | 150% (too fast) |

One row of plots per move: 8 s, 5 s and 4 s.

`'Simple MPC no vel'` is the simple MPC with no velocity term in the stage cost (Dana's idea).


In [ ]:
turn_cases = [(np.pi / 2, 8.0), (np.pi / 2, 5.0), (np.pi / 2, 4.0), (np.pi, 8.0), (np.pi, 5.0), (np.pi, 4.0)]   # one row each: 8 s, 5 s, 4 s
turn_paths = {f"to (3, 3, {'pi/2' if theta_end < 2 else 'pi'}) in {duration:.0f} s":
              path_generator.generate_waypoints_sequence([np.zeros(3), np.array([3.0, 3.0, theta_end])], segment_duration=duration, segment_hold=HOLD_DURATION)
              for theta_end, duration in turn_cases}
turn_controllers = list(WAYPOINT_CONTROLLERS) + ['Simple MPC no vel']     # the same controllers, plus Dana's no-velocity-cost MPC
turn_results = run_and_report("Turning test", turn_paths, turn_controllers)
plot_paths(turn_results, turn_paths, turn_controllers, columns=3)
plot_effort(turn_results, turn_paths, turn_controllers, columns=3)
plot_waypoint_timing(turn_results, turn_paths, turn_controllers, columns=3)
plot_waypoint_closeups(turn_results, turn_paths, turn_controllers)


## 5. Waypoint-executed (Henry): who gets there first

The same controllers with `WaypointExecution`: the controller drives the reference's progress instead of a fixed timeline. At each waypoint the reference waits until the robot has settled there (3 cm, 1 cm/s, 10°; no hold), then starts the next segment. On the figure eights it waits at each of the 7 markers per lap until the robot is within 3 cm (no need to stop), then carries on along the eight. Until then the controller sees the reference only up to that point. Every run bottoms out at the scheduled end plus `WAYPOINT_TIME_ALLOWANCE`: a slow controller still tracks the path but does not reach what it has not reached by then.

Compare with sections 1 to 3 (Dana's fixed time: who is better in the same time).


In [ ]:
executed = WaypointExecution(time_allowance=WAYPOINT_TIME_ALLOWANCE)
executed_paths = {
    "Dana's waypoints (no hold)": PathGeneratorCurvedWithHold(dt=DT_CTRL).generate_waypoints_sequence(WAYPOINTS, segment_duration=5.0, segment_hold=0.0),
    "Dana 8 s (no hold)": path_generator.generate_waypoints_sequence([np.zeros(3), np.array([3.0, 3.0, np.pi / 4])], segment_duration=8.0, segment_hold=0.0),
    "Dana 4 s (too fast, no hold)": path_generator.generate_waypoints_sequence([np.zeros(3), np.array([3.0, 3.0, np.pi / 4])], segment_duration=4.0, segment_hold=0.0),
    "Figure eight 0.4 m (3.4 s)": add_eight_markers(figure_eight_path(0.4, 3.4), 0.4, markers_per_lap=7),
    "Figure eight 0.4 m (2.5 s)": add_eight_markers(figure_eight_path(0.4, 2.5), 0.4, markers_per_lap=7),
    "Figure eight 0.3 m (2.2 s)": add_eight_markers(figure_eight_path(0.3, 2.2), 0.3, markers_per_lap=7),
}
executed_controllers = list(WAYPOINT_CONTROLLERS)     # the same controllers in every test
executed_results = run_and_report("Waypoint-executed (Henry)", executed_paths, executed_controllers, executor=executed)
plot_paths(executed_results, executed_paths, executed_controllers)
plot_errors(executed_results, executed_paths, executed_controllers)
plot_waypoint_timing(executed_results, executed_paths, executed_controllers)
plot_waypoint_closeups(executed_results, executed_paths, executed_controllers)
plot_marker_times(executed_results, executed_paths, executed_controllers)


## UNIT CIRCLE PATH COMPARISON FOR SPEED TEST

Dana's sweep: for each direction and final heading, the target distance grows by `UNITCIRCLE_STEP` until the robot no longer settles at it within one 3 s segment and 1 s hold. MERGED: it runs the merged runner (her call was missing the path).


In [ ]:
#################################################
# REDEFINING THE WAYPOINT TO UNIT CIRCLE ANALYSIS
#################################################

UNITCIRCLE_WP=0.25
UNITCIRCLE_STEP=0.25

THETA_BOUNDS=[-90,90]
THETA_STEP = 20
theta_arry_runner=np.arange(THETA_BOUNDS[0], THETA_BOUNDS[1], THETA_STEP)
ORIENTATION_BOUNDS=[-180,180]
orient_arry_runner=np.arange(ORIENTATION_BOUNDS[0], ORIENTATION_BOUNDS[1], THETA_STEP)

SEGMENT_DURATION = 3.0
HOLD_DURATION = 1.0

####### DEFINING RUNNER FOR ALL POSITIONS 
def run_unitcirlce_benchmark(name):
    print(f"\nExecuting Unit Circle Benchmark for {name} for {SEGMENT_DURATION+HOLD_DURATION}s")
    distances_unit_circle_results = []
    UNITCIRCLE_WP=1

    for theta in theta_arry_runner:
        for orient in orient_arry_runner:
            THETA=theta
            ORIENTATION = orient
            max_dist=None
            
            while max_dist ==None:

                x_wp = UNITCIRCLE_WP*np.cos(np.radians(THETA))
                y_wp = UNITCIRCLE_WP*np.sin(np.radians(THETA))
                t_wp=np.radians(ORIENTATION)
                WAYPOINTS = [
                    np.array([0.0, 0.0, 0.0]),
                    np.array([x_wp, y_wp, t_wp])
                ]

                path_data=PathGenerator_Multi.generate_waypoints_sequence(waypoints=WAYPOINTS,segment_duration=SEGMENT_DURATION,segment_hold=HOLD_DURATION)

                WP_TIMESTAMPS = path_data['segment_boundaries']
                POS_REF = path_data['pos']
                VEL_REF = path_data['vel']
                N_STEPS = len(path_data['pos'])
                T_REF = path_data['t']
                V_LOCAL_REF = path_data['pos_local']

                #print(f"\nExecuting Unit Circle Benchmark for ......... for .....s")
                res = run_simulation(name, path_data)   # MERGED: the merged runner (her fixed timeline)
                
                if res['arrival_times'][1]==None:
                    max_dist=UNITCIRCLE_WP-UNITCIRCLE_STEP
                else:
                    UNITCIRCLE_WP=UNITCIRCLE_WP+UNITCIRCLE_STEP
            distances_unit_circle_results.append(max_dist)
            print(f"Theta: {THETA:<5.1f} | Orient: {ORIENTATION:<5.1f} |Max: {max_dist:<5.2f}")


run_unitcirlce_benchmark('PID')